# Step 14 — Frontend Integration
**Tough Talks · Phase 6**

Goal: connect the vanilla-JS working app (`frontend/app.html` + `app.js` + `app.css`) to the Phase 5 FastAPI surface and prove every wiring path end-to-end. Static frontend is served from the same FastAPI process as the API, at `/app/*`, so the app runs same-origin (no CORS gymnastics) and every endpoint is reachable via a relative URL.

**What ships in this step**:

- `frontend/app.html` — single-page UI with sections for Settings, Person Vault, Talk DNA, Pre-Mortem, Practice Round (chat against `/persona/reply` with rolling history → `/debrief` → optional `/aftermath` → save bundle), Live Mode (audio upload → `/transcribe` + `/emotion/analyze`), and Insights (stored rounds + `/pulse` refresh).
- `frontend/app.js` — vanilla-JS module wrapping every endpoint as `api.*` and managing per-section state. No build step; same-origin `fetch()` calls only.
- `frontend/app.css` — design tokens reused from the concept page (gold/dark).
- `backend/api/main.py` — adds `resolve_frontend_dir()` (env-override `TOUGH_TALKS_FRONTEND_DIR`), a `StaticFiles` mount at `/app`, a `GET /` redirect to `/app/`, and an explicit `GET /app/` route that serves `app.html`. The concept landing stays reachable at `/app/tough_talks_concept.html`.

**What `done` looks like for this step**

1. Static-asset checks — `GET /`, `/app/`, `/app/app.html`, `/app/app.js`, `/app/app.css`, `/app/tough_talks_concept.html` all return 200 with the right content-type; `/app/missing.css` returns 404.
2. Practice flow runs end-to-end against a real Gemma 4 model: build vault → save → analyse TalkDNA → save → pre-mortem → 3 persona-reply turns → debrief → aftermath → save conversation bundle.
3. Pulse runs across 2 saved rounds and the result persists at `/storage/pulse/{person_id}`.
4. Both audio routes (`/transcribe` + `/emotion/analyze`) accept a small gTTS-synthesised clip and return schema-conforming output.
5. Final cell renders a single pass/fail table unconditionally — same shape as Steps 12 / 13.

Single-model strategy (per the constrained-VRAM rule in `knowledge/phases/rules.md`): only the multimodal Gemma 4 variant is loaded. Text routes go through `ModelRegistry.text()`'s fallback to the multimodal pair.

In [1]:
# ── 0. Install / upgrade dependencies ──────────────────────────────────────
# Same rule as Steps 05–12: bump only transformers + accelerate on
# Colab / Kaggle. fastapi / pydantic / multipart / httpx / soundfile /
# librosa / gTTS are all pinned in requirements.txt; Colab pre-installs
# the first four but not gTTS, hence the explicit install.
#
# After this first run, RESTART THE KERNEL before continuing if you
# actually upgraded transformers.

!pip install -q -U transformers accelerate
!pip install -q fastapi 'pydantic>=2.6' python-multipart httpx soundfile librosa gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 93.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ─────────────────────────
# Same shim as Steps 01–13.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ─────────────────────────────────
import io
import json
import tempfile
from pathlib import Path

import torch
from fastapi.testclient import TestClient

from backend.api.deps import ModelRegistry, get_registry, get_storage_root
from backend.api.main import app, resolve_frontend_dir
from backend.core._runtime import (
    ALLOWED_EMOTIONS,
    ALLOWED_HEALTH_TRENDS,
    ALLOWED_MATCH_QUALITIES,
    ALLOWED_RESISTANCE_TYPES,
    DEFAULT_MODEL_ID,
    LoadConfig,
    load_model,
)

In [4]:
# ── 3. Config ──────────────────────────────────
MODEL_ID = DEFAULT_MODEL_ID  # google/gemma-4-E2B-it
STORAGE_ROOT = Path(tempfile.mkdtemp(prefix="step14_storage_"))
FRONTEND_DIR = resolve_frontend_dir()

print("Model              :", MODEL_ID)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device        :", torch.cuda.get_device_name(0))
    print("Free VRAM (GB)     :", round(torch.cuda.mem_get_info()[0] / 1e9, 2))
print("Repo root          :", REPO_ROOT)
print("Storage root       :", STORAGE_ROOT)
print("Frontend dir       :", FRONTEND_DIR, "(exists=%s)" % FRONTEND_DIR.is_dir())

Model              : google/gemma-4-E2B-it
CUDA available     : True
CUDA device        : Tesla T4
Free VRAM (GB)     : 15.53
Repo root          : /content/tough_talks
Storage root       : /tmp/step14_storage_gztmhq1a
Frontend dir       : /content/tough_talks/frontend (exists=True)


In [5]:
# ── 4. Load the multimodal Gemma 4 variant once ───────────────────────────
# Same single-model strategy as Step 12: ONLY the multimodal variant
# fits on a T4 (15.5 GB) without `accelerate` silently offloading
# half the weights to meta-device. Text routes use ModelRegistry.text()'s
# fallback to the multimodal pair.

print("Loading multimodal model ...")
mm_processor, mm_model = load_model(LoadConfig(model_id=MODEL_ID, multimodal=True))
print("  multimodal device     :", mm_model.device)
print("  multimodal dtype      :", next(mm_model.parameters()).dtype)

if torch.cuda.is_available():
    print("Remaining free VRAM   :", round(torch.cuda.mem_get_info()[0] / 1e9, 2), "GB")

Loading multimodal model ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

  multimodal device     : cuda:0
  multimodal dtype      : torch.bfloat16
Remaining free VRAM   : 5.13 GB


In [6]:
# ── 5. Build a TestClient with model + storage + frontend overrides ──────────────
registry = ModelRegistry(
    multimodal_processor=mm_processor,
    multimodal_model=mm_model,
)
app.state.registry = registry
app.state.storage_root = STORAGE_ROOT
app.state.frontend_dir = FRONTEND_DIR
app.dependency_overrides[get_registry] = lambda: registry
app.dependency_overrides[get_storage_root] = lambda: STORAGE_ROOT

client = TestClient(app)
print("TestClient ready.")
print("Frontend dir on state :", app.state.frontend_dir)
print("Storage root on state :", app.state.storage_root)

print("\nRoutes / mounts:")
for r in sorted(app.routes, key=lambda r: getattr(r, 'path', '')):
    methods = ','.join(sorted(getattr(r, 'methods', set()) - {'HEAD'})) or 'MOUNT'
    print(f"  {methods:8s} {getattr(r, 'path', '?')}")

TestClient ready.
Frontend dir on state : /content/tough_talks/frontend
Storage root on state : /tmp/step14_storage_gztmhq1a

Routes / mounts:
  GET      /
  POST     /aftermath
  MOUNT    /app
  GET      /app/
  POST     /debrief
  GET      /docs
  GET      /docs/oauth2-redirect
  POST     /emotion/analyze
  GET      /health
  GET      /openapi.json
  POST     /persona/reply
  POST     /persona/run
  POST     /premortem
  POST     /pulse
  GET      /redoc
  GET      /storage/conversations
  GET      /storage/conversations/{round_id}
  PUT      /storage/conversations/{round_id}
  DELETE   /storage/conversations/{round_id}
  GET      /storage/pulse
  GET      /storage/pulse/{person_id}
  PUT      /storage/pulse/{person_id}
  GET      /storage/talk-dna
  PUT      /storage/talk-dna
  GET      /storage/vault
  GET      /storage/vault/{person_id}
  PUT      /storage/vault/{person_id}
  POST     /talk-dna/analyze
  POST     /transcribe
  POST     /vault/build


In [7]:
# ── 6. Results recorder (renders unconditionally at the end) ─────────────────────
RESULTS: list[dict] = []

def _record(name: str, ok: bool, detail: str) -> None:
    RESULTS.append({"check": name, "ok": ok, "detail": detail})
    print(f"{'PASS' if ok else 'FAIL'}  {name:42s} {detail}")

In [8]:
# ── 7. Static frontend mount checks ───────────────────────────────────────
# Every asset the app shell needs must be reachable at /app/<name>.
# /health surfaces frontend_present=True so the smoke check confirms
# the lifespan / state override is wired correctly.

try:
    r = client.get("/health")
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["status"] == "ok"
    assert body["frontend_present"] is True, body
    _record("/health surfaces frontend", True, f"frontend_dir={body['frontend_dir']!r}")
except Exception as exc:  # noqa: BLE001
    _record("/health surfaces frontend", False, f"{type(exc).__name__}: {exc}")

try:
    r = client.get("/", follow_redirects=False)
    assert r.status_code in (307, 308), r.status_code
    assert r.headers["location"] == "/app/", r.headers
    _record("GET / -> /app/ redirect", True, f"status={r.status_code}")
except Exception as exc:  # noqa: BLE001
    _record("GET / -> /app/ redirect", False, f"{type(exc).__name__}: {exc}")

try:
    r = client.get("/app/")
    assert r.status_code == 200, r.text
    assert "text/html" in r.headers["content-type"], r.headers
    assert "<title>Tough Talks" in r.text, r.text[:200]
    _record("GET /app/ -> app.html", True, f"len={len(r.text)} bytes")
except Exception as exc:  # noqa: BLE001
    _record("GET /app/ -> app.html", False, f"{type(exc).__name__}: {exc}")

for asset, content_check in [
    ("/app/app.html", "<title>Tough Talks"),
    ("/app/app.js",   "const api = {"),
    ("/app/app.css",  "--gold:"),
    ("/app/tough_talks_concept.html", "<title>Tough Talks"),
]:
    try:
        r = client.get(asset)
        assert r.status_code == 200, f"{asset} -> {r.status_code}"
        assert content_check in r.text, f"{asset} missing expected content"
        _record(f"GET {asset}", True, f"len={len(r.text)} bytes")
    except Exception as exc:  # noqa: BLE001
        _record(f"GET {asset}", False, f"{type(exc).__name__}: {exc}")

try:
    r = client.get("/app/does-not-exist.css")
    assert r.status_code == 404, r.text
    _record("GET /app/missing -> 404", True, "StaticFiles 404 as expected")
except Exception as exc:  # noqa: BLE001
    _record("GET /app/missing -> 404", False, f"{type(exc).__name__}: {exc}")

PASS  /health surfaces frontend                  frontend_dir='/content/tough_talks/frontend'
PASS  GET / -> /app/ redirect                    status=307
PASS  GET /app/ -> app.html                      len=8607 bytes
PASS  GET /app/app.html                          len=8607 bytes
PASS  GET /app/app.js                            len=26499 bytes
PASS  GET /app/app.css                           len=9514 bytes
PASS  GET /app/tough_talks_concept.html          len=30932 bytes
PASS  GET /app/missing -> 404                    StaticFiles 404 as expected


In [9]:
# ── 8. Fixtures (Jamie transcript + goal) ────────────────────────────────────
# Same Jamie transcript shape used in Steps 12/13. We feed the
# Step-03 'other'/'text' shape to /vault/build (PersonVault consumer)
# and feed Step-07's 'persona'/'reply' shape to everything downstream.

USER_GOAL = "Get Jamie to commit to Wednesday EOD without re-litigating the missed Tuesday deadline."

VAULT_TURNS = [
    {"speaker": "user",  "text": "I noticed Tuesday's deadline slipped — what's going on?"},
    {"speaker": "other", "text": "I told you the staging tables weren't done last Friday."},
    {"speaker": "user",  "text": "I hear that. I'm not blaming you — I want to land Wednesday EOD together. What's the blocker we can name today?"},
    {"speaker": "other", "text": "The data team hasn't confirmed schema parity. If we agree on a daily Slack check-in I can commit to Wednesday."},
]

print(f"Loaded fixtures: {len(VAULT_TURNS)} turns, goal={USER_GOAL!r}")

Loaded fixtures: 4 turns, goal='Get Jamie to commit to Wednesday EOD without re-litigating the missed Tuesday deadline.'


In [10]:
# ── 9. PersonVault build + save ───────────────────────────────────────────
VAULT_RESULT = None
VAULT_SAVED = None
try:
    r = client.post("/vault/build", json={
        "turns": VAULT_TURNS,
        "name": "Jamie",
        "relationship_type": "colleague",
        "max_new_tokens": 512,
    })
    assert r.status_code == 200, r.text
    VAULT_RESULT = r.json()
    assert VAULT_RESULT["name"] == "Jamie"
    _record("POST /vault/build", True, f"id={VAULT_RESULT['person_id']}, style={VAULT_RESULT['profile']['communication_style']}")
except Exception as exc:  # noqa: BLE001
    _record("POST /vault/build", False, f"{type(exc).__name__}: {exc}")

if VAULT_RESULT is not None:
    try:
        person_id = VAULT_RESULT["person_id"]
        r = client.put(f"/storage/vault/{person_id}", json=VAULT_RESULT)
        assert r.status_code == 200, r.text
        VAULT_SAVED = r.json()["payload"]
        assert VAULT_SAVED["person_id"] == person_id
        list_r = client.get("/storage/vault")
        assert list_r.status_code == 200
        ids = [v["person_id"] for v in list_r.json()["items"]]
        assert person_id in ids, ids
        _record("PUT/GET /storage/vault", True, f"saved + listed (count={list_r.json()['count']})")
    except Exception as exc:  # noqa: BLE001
        _record("PUT/GET /storage/vault", False, f"{type(exc).__name__}: {exc}")

PASS  POST /vault/build                          id=person_f457eedfdcc4, style=assertive
PASS  PUT/GET /storage/vault                     saved + listed (count=1)


In [11]:
# ── 10. TalkDNA analyze + save ──────────────────────────────────────────
TALK_DNA_RESULT = None
try:
    r = client.post("/talk-dna/analyze", json={
        "turns": VAULT_TURNS,
        "user_id": "local",
        "max_new_tokens": 512,
    })
    assert r.status_code == 200, r.text
    TALK_DNA_RESULT = r.json()
    assert 0.0 <= TALK_DNA_RESULT["patterns"]["apology_rate"] <= 1.0
    _record("POST /talk-dna/analyze", True, f"v{TALK_DNA_RESULT['version']}, apology={TALK_DNA_RESULT['patterns']['apology_rate']:.2f}")
except Exception as exc:  # noqa: BLE001
    _record("POST /talk-dna/analyze", False, f"{type(exc).__name__}: {exc}")

if TALK_DNA_RESULT is not None:
    try:
        r = client.put("/storage/talk-dna", json=TALK_DNA_RESULT)
        assert r.status_code == 200, r.text
        got = client.get("/storage/talk-dna", params={"user_id": "local"})
        assert got.status_code == 200
        assert got.json()["version"] == TALK_DNA_RESULT["version"]
        _record("PUT/GET /storage/talk-dna", True, f"user_id=local, v{got.json()['version']}")
    except Exception as exc:  # noqa: BLE001
        _record("PUT/GET /storage/talk-dna", False, f"{type(exc).__name__}: {exc}")

PASS  POST /talk-dna/analyze                     v1, apology=0.00
PASS  PUT/GET /storage/talk-dna                  user_id=local, v1


In [12]:
# ── 11. Pre-Mortem ──────────────────────────────────────────────────────
PREMORTEM_RESULT = None
try:
    r = client.post("/premortem", json={
        "conversation_description": (
            "Tomorrow's 1-on-1 with Jamie about why Tuesday's data-pipeline "
            "deadline slipped again and re-committing to Wednesday EOD."
        ),
        "user_goal": USER_GOAL,
        "person_profile": VAULT_SAVED or VAULT_RESULT or {},
        "talk_dna_profile": TALK_DNA_RESULT or {},
        "enable_thinking": True,
        "max_new_tokens": 2048,
    })
    assert r.status_code == 200, r.text
    PREMORTEM_RESULT = r.json()
    scenarios = PREMORTEM_RESULT["failure_scenarios"]
    assert len(scenarios) == 3
    for i, s in enumerate(scenarios, start=1):
        assert s["scenario_id"] == i
        assert s["simulation_parameters"]["resistance_type"] in ALLOWED_RESISTANCE_TYPES
    _record("POST /premortem", True, f"3 scenarios, types=[{', '.join(s['simulation_parameters']['resistance_type'] for s in scenarios)}]")
except Exception as exc:  # noqa: BLE001
    _record("POST /premortem", False, f"{type(exc).__name__}: {exc}")

PASS  POST /premortem                            3 scenarios, types=[deflect, guilt_trip, counter_attack]


In [13]:
# ── 12. Practice round — 3 user turns via /persona/reply (rolling history) ─────────────
PRACTICE_USER_MESSAGES = [
    "Hey Jamie — I want us to land Wednesday EOD together. Where are we at?",
    "I hear you. I'm not blaming you — what's the one blocker we can name today?",
    "OK — daily Slack check-in starting today, Wednesday EOD locked in. Deal?",
]

ROUND_HISTORY = []
PERSONA_OK = True
for i, msg in enumerate(PRACTICE_USER_MESSAGES, start=1):
    try:
        r = client.post("/persona/reply", json={
            "user_message": msg,
            "persona_profile": VAULT_SAVED or VAULT_RESULT or {},
            "history": list(ROUND_HISTORY),
            "user_goal": USER_GOAL,
            "max_new_tokens": 384,
        })
        assert r.status_code == 200, r.text
        body = r.json()
        assert body["reply"].strip()
        assert body["resistance_type"] in ALLOWED_RESISTANCE_TYPES
        ROUND_HISTORY.append({"speaker": "user", "text": msg})
        ROUND_HISTORY.append({
            "speaker": "persona",
            "persona_name": body["persona_name"],
            "reply": body["reply"],
            "resistance_type": body["resistance_type"],
            "escalation_level": body["escalation_level"],
        })
        print(f"  turn {i}: {body['resistance_type']} @ {body['escalation_level']:.2f}")
    except Exception as exc:  # noqa: BLE001
        PERSONA_OK = False
        print(f"  turn {i} FAIL: {type(exc).__name__}: {exc}")
        break

_record(
    "POST /persona/reply x3 (rolling history)",
    PERSONA_OK and len(ROUND_HISTORY) == 6,
    f"history={len(ROUND_HISTORY)} turns ({len(ROUND_HISTORY)//2} user + {len(ROUND_HISTORY)//2} persona)",
)

  turn 1: deflect @ 0.60
  turn 2: concede @ 0.40
  turn 3: silent @ 0.30
PASS  POST /persona/reply x3 (rolling history)   history=6 turns (3 user + 3 persona)


In [14]:
# ── 13. Debrief + Aftermath ────────────────────────────────────────────────
DEBRIEF_RESULT = None
AFTERMATH_RESULT = None

try:
    r = client.post("/debrief", json={
        "transcript": ROUND_HISTORY,
        "user_goal": USER_GOAL,
        "person_profile": VAULT_SAVED or VAULT_RESULT or {},
        "talk_dna_profile": TALK_DNA_RESULT or {},
        "enable_thinking": True,
        "max_new_tokens": 2048,
    })
    assert r.status_code == 200, r.text
    DEBRIEF_RESULT = r.json()
    assert DEBRIEF_RESULT["one_fix_next_time"].strip()
    _record("POST /debrief", True, (
        f"wins={len(DEBRIEF_RESULT['wins'])}, lost={len(DEBRIEF_RESULT['ground_lost'])}, "
        f"apol={len(DEBRIEF_RESULT['over_apologies'])}, miss={len(DEBRIEF_RESULT['missed_openings'])}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("POST /debrief", False, f"{type(exc).__name__}: {exc}")

try:
    if PREMORTEM_RESULT is None:
        raise RuntimeError("skipped: /premortem did not produce a payload")
    r = client.post("/aftermath", json={
        "premortem": PREMORTEM_RESULT,
        "transcript": ROUND_HISTORY,
        "user_goal": USER_GOAL,
        "person_profile": VAULT_SAVED or VAULT_RESULT or {},
        "debrief": DEBRIEF_RESULT or {},
        "enable_thinking": True,
        "max_new_tokens": 4096,
    })
    assert r.status_code == 200, r.text
    AFTERMATH_RESULT = r.json()
    assert AFTERMATH_RESULT["goal_outcome"]["status"] in {"achieved", "partial", "not_achieved"}
    assert len(AFTERMATH_RESULT["scenario_outcomes"]) == 3
    for outcome in AFTERMATH_RESULT["scenario_outcomes"]:
        assert outcome["match_quality"] in ALLOWED_MATCH_QUALITIES
        assert outcome["materialized"] is (outcome["match_quality"] != "did_not_occur")
    _record("POST /aftermath", True, (
        f"{AFTERMATH_RESULT['goal_outcome']['status']}, accuracy={AFTERMATH_RESULT['prediction_accuracy']:.2f}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("POST /aftermath", False, f"{type(exc).__name__}: {exc}")

PASS  POST /debrief                              wins=3, lost=0, apol=0, miss=0
PASS  POST /aftermath                            achieved, accuracy=0.00


In [15]:
# ── 14. Save the bundled conversation (round 1) ───────────────────────────────
import datetime as _dt

ROUND_ONE_ID = "round_step14_demo_1"
ROUND_ONE_BUNDLE = {
    "round_id": ROUND_ONE_ID,
    "person_id": (VAULT_SAVED or VAULT_RESULT or {}).get("person_id"),
    "person_name": (VAULT_SAVED or VAULT_RESULT or {}).get("name"),
    "started_at": _dt.datetime.now(_dt.timezone.utc).isoformat(),
    "mode": "practice",
    "user_goal": USER_GOAL,
    "transcript": ROUND_HISTORY,
    "premortem": PREMORTEM_RESULT,
    "debrief": DEBRIEF_RESULT,
    "aftermath": AFTERMATH_RESULT,
    "updated_at": _dt.datetime.now(_dt.timezone.utc).isoformat(),
}
ROUND_ONE_BUNDLE = {k: v for k, v in ROUND_ONE_BUNDLE.items() if v is not None}

try:
    r = client.put(f"/storage/conversations/{ROUND_ONE_ID}", json=ROUND_ONE_BUNDLE)
    assert r.status_code == 200, r.text
    got = client.get(f"/storage/conversations/{ROUND_ONE_ID}")
    assert got.status_code == 200
    assert got.json()["round_id"] == ROUND_ONE_ID
    person_id = ROUND_ONE_BUNDLE.get("person_id")
    if person_id:
        filtered = client.get("/storage/conversations", params={"person_id": person_id})
        assert filtered.status_code == 200
        assert any(r["round_id"] == ROUND_ONE_ID for r in filtered.json()["items"]), filtered.json()
    _record("PUT/GET /storage/conversations (round 1)", True, ROUND_ONE_ID)
except Exception as exc:  # noqa: BLE001
    _record("PUT/GET /storage/conversations (round 1)", False, f"{type(exc).__name__}: {exc}")

PASS  PUT/GET /storage/conversations (round 1)   round_step14_demo_1


In [16]:
# ── 15. Synthesise a lighter second round so Pulse has 2 rounds to roll up ─────────────
# Pulse requires N >= 2 rounds. The second round reuses the same
# debrief + aftermath payloads to keep this notebook well under the
# 10-minute Colab budget; the trajectory across rounds isn't the
# subject of step 14 (it's covered in step 11). What we're proving
# here is that two saved rounds correctly drive /pulse and survive
# a round-trip through /storage/pulse.
ROUND_TWO_ID = "round_step14_demo_2"
ROUND_TWO_BUNDLE = {
    "round_id": ROUND_TWO_ID,
    "person_id": (VAULT_SAVED or VAULT_RESULT or {}).get("person_id"),
    "person_name": (VAULT_SAVED or VAULT_RESULT or {}).get("name"),
    "started_at": _dt.datetime.now(_dt.timezone.utc).isoformat(),
    "mode": "practice",
    "user_goal": USER_GOAL,
    "transcript": ROUND_HISTORY,
    "premortem": PREMORTEM_RESULT,
    "debrief": DEBRIEF_RESULT,
    "aftermath": AFTERMATH_RESULT,
    "updated_at": _dt.datetime.now(_dt.timezone.utc).isoformat(),
}
ROUND_TWO_BUNDLE = {k: v for k, v in ROUND_TWO_BUNDLE.items() if v is not None}

try:
    r = client.put(f"/storage/conversations/{ROUND_TWO_ID}", json=ROUND_TWO_BUNDLE)
    assert r.status_code == 200, r.text
    person_id = ROUND_TWO_BUNDLE.get("person_id")
    if person_id:
        listing = client.get("/storage/conversations", params={"person_id": person_id})
        assert listing.status_code == 200
        assert listing.json()["count"] >= 2
    _record("PUT /storage/conversations (round 2)", True, ROUND_TWO_ID)
except Exception as exc:  # noqa: BLE001
    _record("PUT /storage/conversations (round 2)", False, f"{type(exc).__name__}: {exc}")

PASS  PUT /storage/conversations (round 2)       round_step14_demo_2


In [17]:
# ── 16. Pulse over the 2 saved rounds + save ───────────────────────────────────
PULSE_RESULT = None
person_id = (VAULT_SAVED or VAULT_RESULT or {}).get("person_id")

try:
    if person_id is None:
        raise RuntimeError("skipped: no person_id from /vault/build")
    listing = client.get("/storage/conversations", params={"person_id": person_id}).json()
    rounds = [
        {
            "round_id": r["round_id"],
            "started_at": r["started_at"],
            "user_goal": r.get("user_goal"),
            "aftermath": r.get("aftermath"),
            "debrief": r.get("debrief"),
        }
        for r in listing["items"]
        if r.get("aftermath") or r.get("debrief")
    ]
    assert len(rounds) >= 2, f"need 2 rounds with debrief/aftermath, got {len(rounds)}"

    r = client.post("/pulse", json={
        "rounds": rounds,
        "person_profile": VAULT_SAVED or VAULT_RESULT or {},
        "enable_thinking": True,
        "max_new_tokens": 4096,
    })
    assert r.status_code == 200, r.text
    PULSE_RESULT = r.json()
    assert PULSE_RESULT["person_id"] == person_id
    assert PULSE_RESULT["round_count"] == len(rounds)
    assert PULSE_RESULT["health_trend"] in ALLOWED_HEALTH_TRENDS
    _record("POST /pulse", True, (
        f"v{PULSE_RESULT['version']}, trend={PULSE_RESULT['health_trend']}, "
        f"score={PULSE_RESULT['health_score']:.2f}, patterns={len(PULSE_RESULT['recurring_patterns'])}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("POST /pulse", False, f"{type(exc).__name__}: {exc}")

if PULSE_RESULT is not None and person_id is not None:
    try:
        r = client.put(f"/storage/pulse/{person_id}", json=PULSE_RESULT)
        assert r.status_code == 200, r.text
        got = client.get(f"/storage/pulse/{person_id}")
        assert got.status_code == 200
        assert got.json()["version"] == PULSE_RESULT["version"]
        _record("PUT/GET /storage/pulse", True, f"saved v{got.json()['version']} for {person_id}")
    except Exception as exc:  # noqa: BLE001
        _record("PUT/GET /storage/pulse", False, f"{type(exc).__name__}: {exc}")

PASS  POST /pulse                                v1, trend=stable, score=0.90, patterns=1
PASS  PUT/GET /storage/pulse                     saved v1 for person_f457eedfdcc4


In [18]:
# ── 17. Audio routes (synthesised clip) ─────────────────────────────────────
# Same gTTS pattern as Step 12 — in-notebook synthesis stays Kaggle-
# license-safe per `[[project_kaggle_license_safety]]`. Re-encoded
# to mono-16kHz WAV so Gemma 4's audio feature extractor accepts it.

import librosa
import soundfile as sf
from gtts import gTTS

TTS_TEXT = "Hey Jamie I want to land Wednesday end of day together. Let us agree on a daily check-in."
audio_dir = Path(tempfile.mkdtemp(prefix="step14_audio_"))
mp3_path = audio_dir / "clip.mp3"
wav_path = audio_dir / "clip.wav"

tts = gTTS(text=TTS_TEXT, lang="en")
tts.save(str(mp3_path))
wave, sr = librosa.load(str(mp3_path), sr=16000, mono=True)
sf.write(str(wav_path), wave, 16000)
audio_bytes = wav_path.read_bytes()
print(f"Test audio: {wav_path.name} ({len(audio_bytes)} bytes, {len(wave)/16000:.2f} s)")

try:
    r = client.post(
        "/transcribe",
        files={"audio": ("clip.wav", audio_bytes, "audio/wav")},
        data={"max_new_tokens": "128", "language": "en"},
    )
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["transcript"].strip()
    _record("POST /transcribe", True, f"transcript={body['transcript']!r}")
except Exception as exc:  # noqa: BLE001
    _record("POST /transcribe", False, f"{type(exc).__name__}: {exc}")

try:
    r = client.post(
        "/emotion/analyze",
        files={"audio": ("clip.wav", audio_bytes, "audio/wav")},
        data={
            "speaker": "user",
            "turn_id": "step14_t1",
            "transcript_snippet": TTS_TEXT,
            "max_new_tokens": "256",
        },
    )
    assert r.status_code == 200, r.text
    body = r.json()
    results = body["results"]
    assert len(results) >= 1
    assert results[0]["emotions"]["primary"] in ALLOWED_EMOTIONS
    _record("POST /emotion/analyze", True, (
        f"{len(results)} chunk(s), primary={results[0]['emotions']['primary']}, "
        f"intensity={results[0]['emotions']['intensity']:.2f}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("POST /emotion/analyze", False, f"{type(exc).__name__}: {exc}")

Test audio: clip.wav (201260 bytes, 6.29 s)
PASS  POST /transcribe                           transcript='Hey Jamie, I want to land Wednesday end of day together. Let us agree on a daily check-in.'
PASS  POST /emotion/analyze                      1 chunk(s), primary=openness, intensity=0.60


In [19]:
# ── 18. Inspect the on-disk layout ───────────────────────────────────────────
def _tree(root: Path, prefix: str = "") -> None:
    entries = sorted(root.iterdir()) if root.is_dir() else []
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(f"{prefix}{connector}{entry.name}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            _tree(entry, prefix + extension)

print(f"Storage root: {STORAGE_ROOT}")
_tree(STORAGE_ROOT)

Storage root: /tmp/step14_storage_gztmhq1a
├── conversations
│   ├── round_step14_demo_1.json
│   └── round_step14_demo_2.json
├── person_vault
│   └── person_f457eedfdcc4.json
├── pulse
│   └── person_f457eedfdcc4.json
└── talk_dna
    └── local.json


In [20]:
# ── 19. Final pass/fail summary (renders unconditionally) ────────────────────────
if not RESULTS:
    print("No results recorded — the earlier cells did not run.")
else:
    name_w = max(len(r["check"]) for r in RESULTS)
    header = f"  {'check':{name_w}s}  status  detail"
    rule = "  " + "-" * (name_w + 8) + "-" * 60
    print(header)
    print(rule)
    for row in RESULTS:
        mark = "PASS" if row["ok"] else "FAIL"
        print(f"  {row['check']:{name_w}s}  {mark:6s}  {row['detail']}")
    print(rule)
    passed = sum(1 for r in RESULTS if r["ok"])
    total = len(RESULTS)
    print(f"  {passed}/{total} checks OK")

  check                                     status  detail
  ------------------------------------------------------------------------------------------------------------
  /health surfaces frontend                 PASS    frontend_dir='/content/tough_talks/frontend'
  GET / -> /app/ redirect                   PASS    status=307
  GET /app/ -> app.html                     PASS    len=8607 bytes
  GET /app/app.html                         PASS    len=8607 bytes
  GET /app/app.js                           PASS    len=26499 bytes
  GET /app/app.css                          PASS    len=9514 bytes
  GET /app/tough_talks_concept.html         PASS    len=30932 bytes
  GET /app/missing -> 404                   PASS    StaticFiles 404 as expected
  POST /vault/build                         PASS    id=person_f457eedfdcc4, style=assertive
  PUT/GET /storage/vault                    PASS    saved + listed (count=1)
  POST /talk-dna/analyze                    PASS    v1, apology=0.00
  PUT/GET /stor

In [ ]:
# ── Migrate saved bundles through the silent-demotion coercer ─────────────
# Step 14 surfaced a multi-sentence persona reply mislabelled ``silent``
# (turn 3 of the practice round: "Deal. I'll send the plan over by
# tomorrow morning."). The runtime now enforces Step 07's "silent is
# for one-word withdrawals" rule in code via
# ``_demote_silent_for_long_reply``. This cell re-coerces the
# already-saved bundles in place — pure deterministic post-processing,
# no model call needed.
#
# Same defence-in-depth shape as Step 09's ``_APOLOGY_CUE_RE`` and
# Step 10's ``_TRANSCRIPT_META_PREFIX_RE``: prompt teaches the model,
# code enforces the contract, and a one-off migration heals existing
# saved data without re-paying the model cost.
#
# After this cell runs, re-running the inspection cell above should
# show turn 3 as ``concede`` (matched by the "Deal." signal).

# The Step 14 runtime fix is fresh — reload persona_sim so the helper
# is importable even if the kernel cached the pre-fix module.
import importlib
import backend.core._runtime.persona_sim as _persona_sim_mod
importlib.reload(_persona_sim_mod)
from backend.core._runtime.persona_sim import _demote_silent_for_long_reply

migrated = 0
for round_id in (ROUND_ONE_ID, ROUND_TWO_ID):
    got = client.get(f"/storage/conversations/{round_id}")
    assert got.status_code == 200, got.text
    bundle = got.json()
    changed = False
    for turn in bundle.get("transcript", []):
        if turn.get("speaker") != "persona":
            continue
        old_rt = turn.get("resistance_type")
        new_rt = _demote_silent_for_long_reply(old_rt, turn.get("reply", ""))
        if old_rt != new_rt:
            print(f"  {round_id}: {old_rt} -> {new_rt}")
            print(f"    reply: {turn.get('reply', '')}")
            turn["resistance_type"] = new_rt
            changed = True
            migrated += 1
    if changed:
        put = client.put(f"/storage/conversations/{round_id}", json=bundle)
        assert put.status_code == 200, put.text

if migrated == 0:
    print("\n0 turns needed demotion — saved bundles are already clean.")
else:
    print(f"\n{migrated} persona turn(s) demoted across 2 saved rounds. Re-run the inspection cell to verify.")